# Sparse-Structure Mapper Group-Action Evaluation

Evaluate released sparse-structure mappers on every dumped toys4k mesh. Identity, inverse, composition, and six 60-degree rotations are generated deterministically from each shape id. Prediction-to-prediction comparisons measure feature cosine consistency. Prediction-to-GT comparisons additionally measure feature errors and decoded occupancy agreement.

In [1]:
import hashlib
import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import trellis2.models as trellis2_models
from o_voxel.convert.flexible_dual_grid import intersect_occ
from pytorch3d.transforms import axis_angle_to_matrix, quaternion_to_matrix
from tqdm.auto import tqdm

from symtrellis.geometry import t_abs2grid
from symtrellis.mapper import from_pretrained

torch.set_grad_enabled(False)

MAPPER_NAMES = [
    "trellis2/sparse_structure/neighbor_graph/finetune",
    "trellis2/sparse_structure/swin3d/legacy",
]

DUMPED_MESH_DIR = Path("/mnt/scratch/trellis500k/toys4k/trellis2/dumped_mesh")
SS_ENCODER_ID = "microsoft/TRELLIS-image-large/ckpts/ss_enc_conv3d_16l8_fp16"
SS_DECODER_ID = "microsoft/TRELLIS-image-large/ckpts/ss_dec_conv3d_16l8_fp16"

DEVICE = torch.device("cuda:0")
GLOBAL_SEED = 20260701
NUM_RANDOM_TRIALS = 2
CLOSURE_STEPS = 6
SS_EVAL_BATCH_SIZE = 16
DST_INPUT_NORM_THRESHOLD = 1.5
LATENT_GRID_SIZE = 16
OCCUPANCY_GRID_SIZE = 64
AABB = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]]

In [2]:
mesh_paths = sorted(DUMPED_MESH_DIR.glob('*.pickle'))
print(f'dumped meshes: {len(mesh_paths):,}')

ss_encoder = trellis2_models.from_pretrained(SS_ENCODER_ID).eval().to(DEVICE)
ss_decoder = trellis2_models.from_pretrained(SS_DECODER_ID).eval().to(DEVICE)
ss_decoder.convert_to_fp32()
ss_decoder.float()

for parameter in ss_encoder.parameters():
    parameter.requires_grad_(False)
for parameter in ss_decoder.parameters():
    parameter.requires_grad_(False)

grid_xyz = torch.stack(
    torch.meshgrid(
        torch.arange(LATENT_GRID_SIZE, device=DEVICE),
        torch.arange(LATENT_GRID_SIZE, device=DEVICE),
        torch.arange(LATENT_GRID_SIZE, device=DEVICE),
        indexing='ij',
    ),
    dim=-1,
).reshape(-1, 3).to(dtype=torch.int32)
grid_positions = (grid_xyz.float() + 0.5) / LATENT_GRID_SIZE - 0.5


dumped meshes: 3,229
[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn


In [3]:
def rotation_for_trial(shape_id, operation, trial_id, rotation_id):
    key = f'{GLOBAL_SEED}:{shape_id}:{operation}:{trial_id}:{rotation_id}'
    seed = int.from_bytes(hashlib.sha256(key.encode('ascii')).digest()[:8], 'big')
    generator = torch.Generator(device='cpu')
    generator.manual_seed(seed)

    if operation == 'closure':
        axis = torch.randn(3, generator=generator)
        axis = axis / axis.norm().clamp_min(1e-12)
        angle = 2.0 * math.pi / CLOSURE_STEPS
        return axis_angle_to_matrix((axis * angle)[None])[0].to(DEVICE)

    quaternion = torch.randn(4, generator=generator)
    quaternion = quaternion / quaternion.norm().clamp_min(1e-12)
    return quaternion_to_matrix(quaternion[None])[0].to(DEVICE)


@torch.no_grad()
def encode_ss_mesh(mesh, rotation, encoder):
    vertices, faces = mesh
    rotated_vertices = vertices @ rotation.T
    occupied_coords = intersect_occ(
        vertices=rotated_vertices.contiguous(),
        faces=faces,
        grid_size=OCCUPANCY_GRID_SIZE,
        aabb=AABB,
    ).long()
    occupancy = torch.zeros(
        (1, OCCUPANCY_GRID_SIZE, OCCUPANCY_GRID_SIZE, OCCUPANCY_GRID_SIZE),
        device=DEVICE,
        dtype=torch.float32,
    )
    occupancy[0, occupied_coords[:, 0], occupied_coords[:, 1], occupied_coords[:, 2]] = 1.0
    return encoder(occupancy[None], sample_posterior=False)[0].float().contiguous()


@torch.no_grad()
def apply_ss_mapper(model, latent, source_pose, destination_pose):
    batch_size = latent.shape[0]
    # Mapper transforms are destination-to-source: R_src @ R_dst.T.
    O_dst2src = torch.bmm(source_pose, destination_pose.transpose(1, 2)).float()
    zero_translation = torch.zeros((batch_size, 3), device=DEVICE, dtype=torch.float32)

    source_coords = []
    destination_coords = []
    source_features = []
    for sample_id in range(batch_size):
        O_sample = O_dst2src[sample_id]
        destination_in_source = grid_positions @ O_sample.T
        source_in_destination = grid_positions @ O_sample
        destination_mask = torch.all((destination_in_source >= -0.5) & (destination_in_source <= 0.5), dim=1)
        source_mask = torch.all((source_in_destination >= -0.5) & (source_in_destination <= 0.5), dim=1)
        source_xyz = grid_xyz[source_mask]
        destination_xyz = grid_xyz[destination_mask]
        source_batch = torch.full((source_xyz.shape[0], 1), sample_id, device=DEVICE, dtype=torch.int32)
        destination_batch = torch.full((destination_xyz.shape[0], 1), sample_id, device=DEVICE, dtype=torch.int32)
        source_coords.append(torch.cat([source_batch, source_xyz], dim=1))
        destination_coords.append(torch.cat([destination_batch, destination_xyz], dim=1))
        source_features.append(
            latent[
                sample_id,
                :,
                source_xyz[:, 0].long(),
                source_xyz[:, 1].long(),
                source_xyz[:, 2].long(),
            ].T
        )

    coords_src = torch.cat(source_coords, dim=0).contiguous()
    coords_dst = torch.cat(destination_coords, dim=0).contiguous()
    feats_src = torch.cat(source_features, dim=0).contiguous()
    coeff = model(
        coords_src=coords_src,
        coords_dst=coords_dst,
        O_dst2src=O_dst2src,
        t_dst2src=t_abs2grid(zero_translation, O_dst2src, LATENT_GRID_SIZE),
        s_dst2src=torch.ones(batch_size, device=DEVICE, dtype=torch.long),
    ).to(device=DEVICE, dtype=torch.float32)
    mapped = coeff.apply(feats_src.to(dtype=coeff.dtype)).float()
    destination_has_edge = torch.bincount(coeff.e_ids_dst, minlength=coeff.num_dst) > 0
    written_coords = coords_dst[destination_has_edge]
    written_features = mapped[destination_has_edge]

    # Match training-time scatter semantics: destinations without an edge remain zero.
    output = latent.new_zeros(latent.shape)
    output[
        written_coords[:, 0].long(),
        :,
        written_coords[:, 1].long(),
        written_coords[:, 2].long(),
        written_coords[:, 3].long(),
    ] = written_features
    return output


In [4]:
def compute_ss_consistency_metrics(lhs, rhs, endpoint_gt):
    batch_size, channels = lhs.shape[:2]
    lhs_rows = lhs.permute(0, 2, 3, 4, 1).reshape(batch_size, -1, channels).float()
    rhs_rows = rhs.permute(0, 2, 3, 4, 1).reshape(batch_size, -1, channels).float()
    gt_rows = endpoint_gt.permute(0, 2, 3, 4, 1).reshape(batch_size, -1, channels).float()
    cosine = F.cosine_similarity(lhs_rows, rhs_rows, dim=2)
    gt_norm = gt_rows.norm(dim=2)
    # The endpoint encoder GT defines a common mask and weight for both paths.
    filtered = gt_norm >= DST_INPUT_NORM_THRESHOLD
    filtered_cosine = (cosine * filtered).sum(dim=1) / filtered.sum(dim=1)
    norm_sum = gt_norm.sum(dim=1).clamp_min(1e-12)
    return {
        'feature_filtered_cosine': filtered_cosine,
        'feature_unfiltered_cosine': cosine.mean(dim=1),
        'feature_norm_weighted_cosine': (cosine * gt_norm).sum(dim=1) / norm_sum,
    }


@torch.no_grad()
def compute_ss_gt_metrics(prediction, target, target_decoder_logits, decoder):
    batch_size, channels = prediction.shape[:2]
    prediction_rows = prediction.permute(0, 2, 3, 4, 1).reshape(batch_size, -1, channels).float()
    target_rows = target.permute(0, 2, 3, 4, 1).reshape(batch_size, -1, channels).float()
    difference = prediction_rows - target_rows
    row_l1 = difference.abs().mean(dim=2)
    row_l2 = difference.square().mean(dim=2)
    row_cosine = F.cosine_similarity(prediction_rows, target_rows, dim=2)
    target_norm = target_rows.norm(dim=2)
    filtered = target_norm >= DST_INPUT_NORM_THRESHOLD
    filtered_count = filtered.sum(dim=1)
    target_norm_sum = target_norm.sum(dim=1).clamp_min(1e-12)
    filtered_cosine = (row_cosine * filtered).sum(dim=1) / filtered_count
    unfiltered_cosine = row_cosine.mean(dim=1)
    weighted_cosine = (row_cosine * target_norm).sum(dim=1) / target_norm_sum

    prediction_logits = decoder(prediction).float()
    decoder_l2 = (prediction_logits - target_decoder_logits).square().flatten(1).mean(dim=1)
    prediction_occupied = prediction_logits > 0
    target_occupied = target_decoder_logits > 0
    intersection = (prediction_occupied & target_occupied).flatten(1).sum(dim=1).float()
    union = (prediction_occupied | target_occupied).flatten(1).sum(dim=1).float()

    return {
        'feature_filtered_l1': (row_l1 * filtered).sum(dim=1) / filtered_count,
        'feature_filtered_l2': (row_l2 * filtered).sum(dim=1) / filtered_count,
        'feature_filtered_cosine': filtered_cosine,
        'feature_filtered_cosine_distance': 1.0 - filtered_cosine,
        'feature_filter_keep_ratio': filtered.float().mean(dim=1),
        'feature_unfiltered_l1': row_l1.mean(dim=1),
        'feature_unfiltered_l2': row_l2.mean(dim=1),
        'feature_unfiltered_cosine': unfiltered_cosine,
        'feature_unfiltered_cosine_distance': 1.0 - unfiltered_cosine,
        'feature_norm_weighted_l1': (row_l1 * target_norm).sum(dim=1) / target_norm_sum,
        'feature_norm_weighted_l2': (row_l2 * target_norm).sum(dim=1) / target_norm_sum,
        'feature_norm_weighted_cosine': weighted_cosine,
        'feature_norm_weighted_cosine_distance': 1.0 - weighted_cosine,
        'decoder_l2': decoder_l2,
        'decoder_iou': intersection / union.clamp_min(1.0),
    }


def accumulate_metrics(metric_sums, metric_counts, prefix, metrics):
    for name, values in metrics.items():
        key = f'{prefix}_{name}'
        metric_sums[key] = metric_sums.get(
            key, values.new_zeros((), dtype=torch.float64)
        ) + values.double().sum()
        metric_counts[key] = metric_counts.get(key, 0) + values.shape[0]


In [5]:
@torch.no_grad()
def evaluate_ss_group_actions(mesh_paths, model, encoder, decoder):
    metric_sums = {}
    metric_counts = {}
    model.eval()

    for batch_start in tqdm(
        range(0, len(mesh_paths), SS_EVAL_BATCH_SIZE),
        desc='evaluate sparse structure',
        leave=False,
    ):
        batch_paths = mesh_paths[batch_start:batch_start + SS_EVAL_BATCH_SIZE]
        shape_ids = [path.stem for path in batch_paths]
        meshes = []
        for path in batch_paths:
            with path.open('rb') as file:
                dump = pickle.load(file)
            vertex_arrays = []
            face_arrays = []
            vertex_offset = 0
            for obj in dump['objects']:
                if obj['vertices'].size == 0 or obj['faces'].size == 0:
                    continue
                vertex_arrays.append(obj['vertices'])
                face_arrays.append(obj['faces'] + vertex_offset)
                vertex_offset += len(obj['vertices'])
            vertices = torch.from_numpy(np.concatenate(vertex_arrays, axis=0)).to(DEVICE, dtype=torch.float32)
            faces = torch.from_numpy(np.concatenate(face_arrays, axis=0)).to(DEVICE, dtype=torch.long)
            meshes.append((vertices.contiguous(), faces.contiguous()))

        batch_size = len(meshes)
        identity_pose = torch.eye(3, device=DEVICE).expand(batch_size, -1, -1).clone()
        identity_gt = torch.stack(
            [encode_ss_mesh(mesh, identity_pose[sample_id], encoder) for sample_id, mesh in enumerate(meshes)]
        )
        identity_target_logits = decoder(identity_gt).float()
        identity_prediction = apply_ss_mapper(model, identity_gt, identity_pose, identity_pose)
        accumulate_metrics(
            metric_sums,
            metric_counts,
            'identity_gt',
            compute_ss_gt_metrics(identity_prediction, identity_gt, identity_target_logits, decoder),
        )

        for trial_id in range(NUM_RANDOM_TRIALS):
            inverse_pose = torch.stack(
                [rotation_for_trial(shape_id, 'inverse', trial_id, 0) for shape_id in shape_ids]
            )
            inverse_forward = apply_ss_mapper(model, identity_gt, identity_pose, inverse_pose)
            inverse_prediction = apply_ss_mapper(model, inverse_forward, inverse_pose, identity_pose)
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'inverse_consistency',
                compute_ss_consistency_metrics(inverse_prediction, identity_prediction, identity_gt),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'inverse_gt',
                compute_ss_gt_metrics(inverse_prediction, identity_gt, identity_target_logits, decoder),
            )

            first_pose = torch.stack(
                [rotation_for_trial(shape_id, 'composition', trial_id, 0) for shape_id in shape_ids]
            )
            second_rotation = torch.stack(
                [rotation_for_trial(shape_id, 'composition', trial_id, 1) for shape_id in shape_ids]
            )
            final_pose = torch.bmm(second_rotation, first_pose)
            composition_gt = torch.stack(
                [encode_ss_mesh(mesh, final_pose[sample_id], encoder) for sample_id, mesh in enumerate(meshes)]
            )
            composition_target_logits = decoder(composition_gt).float()
            first_prediction = apply_ss_mapper(model, identity_gt, identity_pose, first_pose)
            sequential_prediction = apply_ss_mapper(model, first_prediction, first_pose, final_pose)
            direct_prediction = apply_ss_mapper(model, identity_gt, identity_pose, final_pose)
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_consistency',
                compute_ss_consistency_metrics(sequential_prediction, direct_prediction, composition_gt),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_sequential_gt',
                compute_ss_gt_metrics(sequential_prediction, composition_gt, composition_target_logits, decoder),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_direct_gt',
                compute_ss_gt_metrics(direct_prediction, composition_gt, composition_target_logits, decoder),
            )

            closure_rotation = torch.stack(
                [rotation_for_trial(shape_id, 'closure', trial_id, 0) for shape_id in shape_ids]
            )
            closure_prediction = identity_gt
            source_pose = identity_pose
            for closure_step in range(CLOSURE_STEPS):
                destination_pose = torch.bmm(closure_rotation, source_pose)
                if closure_step == CLOSURE_STEPS - 1:
                    destination_pose = identity_pose
                closure_prediction = apply_ss_mapper(
                    model, closure_prediction, source_pose, destination_pose
                )
                source_pose = destination_pose
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'closure_consistency',
                compute_ss_consistency_metrics(closure_prediction, identity_prediction, identity_gt),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'closure_gt',
                compute_ss_gt_metrics(closure_prediction, identity_gt, identity_target_logits, decoder),
            )

    return {name: (value / metric_counts[name]).item() for name, value in metric_sums.items()}


In [6]:
records = []
for model_name in tqdm(MAPPER_NAMES, desc='models'):
    print(f'Evaluating {model_name}')
    mapper = from_pretrained(model_name, device=DEVICE).eval()
    metrics = evaluate_ss_group_actions(mesh_paths, mapper, ss_encoder, ss_decoder)
    records.append({'model_name': model_name, **metrics})
    del mapper
    torch.cuda.empty_cache()

evaluation_df = pd.DataFrame(records)
evaluation_df


models:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating trellis2/sparse_structure/neighbor_graph/finetune


evaluate sparse structure:   0%|          | 0/202 [00:00<?, ?it/s]

Evaluating trellis2/sparse_structure/swin3d/legacy


evaluate sparse structure:   0%|          | 0/202 [00:00<?, ?it/s]

,model_name,identity_gt_feature_filtered_l1,identity_gt_feature_filtered_l2,identity_gt_feature_filtered_cosine,identity_gt_feature_filtered_cosine_distance,identity_gt_feature_filter_keep_ratio,identity_gt_feature_unfiltered_l1,identity_gt_feature_unfiltered_l2,identity_gt_feature_unfiltered_cosine,identity_gt_feature_unfiltered_cosine_distance,...,closure_gt_feature_unfiltered_l1,closure_gt_feature_unfiltered_l2,closure_gt_feature_unfiltered_cosine,closure_gt_feature_unfiltered_cosine_distance,closure_gt_feature_norm_weighted_l1,closure_gt_feature_norm_weighted_l2,closure_gt_feature_norm_weighted_cosine,closure_gt_feature_norm_weighted_cosine_distance,closure_gt_decoder_l2,closure_gt_decoder_iou
0,trellis2/sparse_structure/neighbor_graph/finetune,0.211178,0.094235,0.958704,0.041296,0.183696,0.068594,0.023984,0.841123,0.158877,...,0.166736,0.122455,0.223264,0.776736,0.457717,0.428909,0.514345,0.485655,536.514575,0.669065
1,trellis2/sparse_structure/swin3d/legacy,0.215641,0.092636,0.957626,0.042374,0.183696,0.066082,0.022237,0.882869,0.117131,...,0.195135,0.169353,0.187392,0.812608,0.549653,0.599495,0.416118,0.583882,878.875933,0.529291
